## 07_lenormand_pytdlm_cz
  Same comparison with PyTDLM: three laws x four constraint levels, exponents calibrated by
  maximizing CPC, 100 realizations; cross-check of the scikit-mobility realizations.
  Output: data/processed/07_lenormand_pytdlm_cz/. Environment: nuts3_tdlm.

In [ ]:
from importlib.metadata import version
import numpy as np
from TDLM import tdlm

print("PyTDLM version:", version("PyTDLM"))

# Tiny synthetic example (3 regions) from the package README
mi = np.array([100, 200, 150])           # origin masses
mj = np.array([80, 180, 120])            # destination masses
dij = np.array([[0, 10, 15], [10, 0, 8], [15, 8, 0]])
Oi = np.array([50, 80, 60])              # out-trips
Dj = np.array([40, 90, 50])              # in-trips
obs = np.array([[0, 25, 25], [30, 0, 50], [35, 35, 0]])

sim = tdlm.run_law_model(law="NGravExp", mass_origin=mi, mass_destination=mj,
                         distance=dij, exponent=np.array([0.1, 0.2]),
                         model="DCM", out_trips=Oi, in_trips=Dj, repli=5)
res = tdlm.gof(sim=sim, obs=obs, distance=dij)
print(res[0.1].round(3).to_string(index=False))

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
COUNTRY = "CZ"
COL_ID, COL_POP = "nuts3_id", "population"
SHARED_DIR = config.INTERIM_DIR / "lenormand_comparison" / COUNTRY.lower()
POP_CSV = SHARED_DIR / "population.csv"
DIST_CSV = SHARED_DIR / "distance_km.csv"
OD_CSV = SHARED_DIR / "od_observed.csv"
# ------------------------------------------------------------------

pop = pd.read_csv(POP_CSV)
dist_df = pd.read_csv(DIST_CSV, index_col=0)
od_df = pd.read_csv(OD_CSV, index_col=0)

ids = pop[COL_ID].tolist()
assert list(dist_df.index) == ids == list(dist_df.columns), "distance matrix order differs"
assert list(od_df.index) == ids == list(od_df.columns), "OD matrix order differs"

mass = pop[COL_POP].to_numpy(dtype=float)
distance = dist_df.to_numpy(dtype=float)
obs = od_df.to_numpy(dtype=float).copy()
np.fill_diagonal(obs, 0)                     # T_ii = 0, as in the paper
out_trips = obs.sum(axis=1)                  # O_i
in_trips = obs.sum(axis=0)                   # D_j

print("Regions:", len(ids))
print("Shapes: mass", mass.shape, "| distance", distance.shape, "| obs", obs.shape)
print("Total commuters (off-diagonal):", int(obs.sum()))
print("Sum of out_trips equals sum of in_trips:", bool(np.isclose(out_trips.sum(), in_trips.sum())))
print("Zero marginals:", int((out_trips == 0).sum()), "out |", int((in_trips == 0).sum()), "in")
print("Loaded from:", os.path.relpath(SHARED_DIR))

In [ ]:
import pandas as pd
from TDLM import tdlm

# ----------------------------- CONFIG -----------------------------
OUT_DIR = config.PROCESSED_DIR / "07_lenormand_pytdlm_cz"
CALIB_CSV = OUT_DIR / "calibrated_exponents.csv"
CALIB_LAWS = ["NGravExp", "NGravPow"]          # Rad has no parameter
CONSTRAINT_LEVELS = ["UM", "PCM", "ACM", "DCM"]
N_REPS_CALIB = 10
SEED = 0
CALIB_MEASURE = "CPC"
# ------------------------------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)

if CALIB_CSV.exists():
    calib = pd.read_csv(CALIB_CSV)
    print("Loaded from checkpoint:", os.path.relpath(CALIB_CSV))
else:
    calib = tdlm.run_optimization(
        mass_origin=mass, mass_destination=mass, distance=distance, obs=obs,
        law=CALIB_LAWS, model=CONSTRAINT_LEVELS,
        out_trips=out_trips, in_trips=in_trips,
        repli=N_REPS_CALIB, measure=CALIB_MEASURE, random_seed=SEED,
    )
    calib.to_csv(CALIB_CSV, index=False)
    print("Saved:", os.path.relpath(CALIB_CSV))

print(calib.round(4).to_string(index=False))

In [ ]:
import io
import warnings
from contextlib import redirect_stdout
import numpy as np
import pandas as pd
from TDLM import tdlm

# ----------------------------- CONFIG -----------------------------
N_REPS = 100
SEED_EVAL = 1
LAWS = ["NGravExp", "NGravPow", "Rad"]
CONSTRAINT_LEVELS = ["UM", "PCM", "ACM", "DCM"]
METRIC_COLS = ["CPC", "CPL", "CPCd", "KL_div", "RMSE"]    # KS columns are not used
RUNS_DIR = OUT_DIR / "runs"
# ------------------------------------------------------------------

RUNS_DIR.mkdir(parents=True, exist_ok=True)
# Zero diagonal distance gives harmless inf in the radiation law; keep file paths out of the output
warnings.filterwarnings("ignore", category=RuntimeWarning)

def first_value(x):
    """Return the object itself or the first item of a dict keyed by exponent."""
    return next(iter(x.values())) if isinstance(x, dict) else x

for law in LAWS:
    for level in CONSTRAINT_LEVELS:
        tag = f"{law}_{level}"
        sim_path, met_path = RUNS_DIR / f"sim_{tag}.npy", RUNS_DIR / f"metrics_{tag}.csv"
        if sim_path.exists() and met_path.exists():
            print("Skip (checkpoint):", tag)
            continue
        if law == "Rad":
            exponent = 1.0                       # ignored by the radiation law
        else:
            row = calib[(calib["Law"] == law) & (calib["Model"] == level)].iloc[0]
            exponent = np.array([row["Exponent"]])
        with redirect_stdout(io.StringIO()):     # silence the package's own progress prints
            sim = tdlm.run_law_model(
                law=law, mass_origin=mass, mass_destination=mass, distance=distance,
                exponent=exponent, model=level, out_trips=out_trips, in_trips=in_trips,
                repli=N_REPS, random_seed=SEED_EVAL, verbose=False)
            res = tdlm.gof(sim=sim, obs=obs, distance=distance)
        met = first_value(res)[["Replication"] + METRIC_COLS]
        np.save(sim_path, np.asarray(first_value(sim)))
        met.to_csv(met_path, index=False)
        print("Done:", tag, "| CPC mean:", round(met["CPC"].mean(), 4))

In [ ]:
import pandas as pd

# ----------------------------- CONFIG -----------------------------
BY_REP_CSV = OUT_DIR / "metrics_pytdlm_by_rep.csv"
SUMMARY_CSV = OUT_DIR / "metrics_pytdlm_summary.csv"
# ------------------------------------------------------------------

frames = []
for law in LAWS:
    for level in CONSTRAINT_LEVELS:
        m = pd.read_csv(RUNS_DIR / f"metrics_{law}_{level}.csv")
        m.insert(0, "Model", level)
        m.insert(0, "Law", law)
        frames.append(m)
by_rep = pd.concat(frames, ignore_index=True)

summary = (by_rep.groupby(["Law", "Model"], sort=False)[METRIC_COLS]
           .agg(["mean", "std"]))
summary.columns = [f"{a}_{b}" for a, b in summary.columns]
summary = summary.reset_index()

by_rep.to_csv(BY_REP_CSV, index=False)
summary.to_csv(SUMMARY_CSV, index=False)
print("Saved:", os.path.relpath(BY_REP_CSV), by_rep.shape)
print("Saved:", os.path.relpath(SUMMARY_CSV), summary.shape)
print(summary[["Law", "Model", "CPC_mean", "CPL_mean", "CPCd_mean", "KL_div_mean", "RMSE_mean"]]
      .round(4).to_string(index=False))

In [ ]:
import io
from contextlib import redirect_stdout
import numpy as np
import pandas as pd
from TDLM import tdlm

# ----------------------------- CONFIG -----------------------------
SKMOB_DIR = config.PROCESSED_DIR / "06_lenormand_skmob_cz"
SKMOB_MODELS = ["gravity_exp_UM", "gravity_exp_PCM",
                "gravity_power_UM", "gravity_power_PCM", "radiation_PCM"]
XCHECK_BY_REP_CSV = OUT_DIR / "metrics_skmob_in_pytdlm_by_rep.csv"
XCHECK_SUMMARY_CSV = OUT_DIR / "metrics_skmob_in_pytdlm_summary.csv"
# ------------------------------------------------------------------

if XCHECK_BY_REP_CSV.exists() and XCHECK_SUMMARY_CSV.exists():
    xs = pd.read_csv(XCHECK_SUMMARY_CSV)
    print("Loaded from checkpoint:", os.path.relpath(XCHECK_SUMMARY_CSV))
else:
    frames = []
    for name in SKMOB_MODELS:
        sim = np.load(SKMOB_DIR / f"realizations_{name}.npy")      # shape (repli, n, n)
        with redirect_stdout(io.StringIO()):
            res = tdlm.gof(sim=sim, obs=obs, distance=distance)
        m = first_value(res)[["Replication"] + METRIC_COLS].copy()
        m.insert(0, "SkmobModel", name)
        frames.append(m)
    xrep = pd.concat(frames, ignore_index=True)
    xs = xrep.groupby("SkmobModel", sort=False)[METRIC_COLS].agg(["mean", "std"])
    xs.columns = [f"{a}_{b}" for a, b in xs.columns]
    xs = xs.reset_index()
    xrep.to_csv(XCHECK_BY_REP_CSV, index=False)
    xs.to_csv(XCHECK_SUMMARY_CSV, index=False)
    print("Saved:", os.path.relpath(XCHECK_BY_REP_CSV), xrep.shape)
    print("Saved:", os.path.relpath(XCHECK_SUMMARY_CSV), xs.shape)

print(xs[["SkmobModel", "CPC_mean", "CPL_mean", "CPCd_mean", "KL_div_mean", "RMSE_mean"]]
      .round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG_NAME = "cz_lenormand_cpc_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
LAW_STYLE = {                       # color, marker, label
    "NGravExp": ("#2a78d6", "o", "NGravExp"),
    "NGravPow": ("#eb6834", "s", "NGravPow"),
    "Rad":      ("#1baf7a", "^", "Rad (Simini)"),
}
Y_OFFSET = {"NGravExp": 0.18, "NGravPow": 0.0, "Rad": -0.18}   # avoid overlapping marks
X_PAD = 0.05
# ------------------------------------------------------------------

levels = CONSTRAINT_LEVELS
ypos = {lvl: len(levels) - i for i, lvl in enumerate(levels)}   # UM on top

fig, ax = plt.subplots(figsize=(7, 4.2))
for law, (color, marker, label) in LAW_STYLE.items():
    sub = summary[summary["Law"] == law]
    ax.scatter(sub["CPC_mean"], [ypos[m] + Y_OFFSET[law] for m in sub["Model"]],
               color=color, marker=marker, s=60, edgecolor="white", linewidth=1.2,
               label=label, zorder=3)
ax.set_yticks(list(ypos.values()))
ax.set_yticklabels(list(ypos.keys()))
ax.set_xlim(summary["CPC_mean"].min() - X_PAD, min(1.0, summary["CPC_mean"].max() + X_PAD))
ax.set_xlabel("CPC (mean of replications)")
ax.set_title("CZ kraj level: CPC by law and constraint level (PyTDLM)")
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3)
fig.tight_layout()

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG_NAME))
plt.show()

In [ ]:
import io
from contextlib import redirect_stdout

# ----------------------------- CONFIG -----------------------------
RAND_LAW = "Rand"
# ------------------------------------------------------------------

rows = []
for level in CONSTRAINT_LEVELS:
    tag = f"{RAND_LAW}_{level}"
    sim_path, met_path = RUNS_DIR / f"sim_{tag}.npy", RUNS_DIR / f"metrics_{tag}.csv"
    if not (sim_path.exists() and met_path.exists()):
        with redirect_stdout(io.StringIO()):     # silence the package's own progress prints
            sim = tdlm.run_law_model(
                law=RAND_LAW, mass_origin=mass, mass_destination=mass, distance=distance,
                exponent=1.0, model=level, out_trips=out_trips, in_trips=in_trips,
                repli=N_REPS, random_seed=SEED_EVAL, verbose=False)
            res = tdlm.gof(sim=sim, obs=obs, distance=distance)
        first_value(res)[["Replication"] + METRIC_COLS].to_csv(met_path, index=False)
        np.save(sim_path, np.asarray(first_value(sim)))
        print("Done:", tag)
    else:
        print("Skip (checkpoint):", tag)
    m = pd.read_csv(met_path)
    row = {"Law": RAND_LAW, "Model": level}
    for c in METRIC_COLS:
        row[f"{c}_mean"], row[f"{c}_std"] = m[c].mean(), m[c].std()
    rows.append(row)

# append to the in-memory summary (replace older Rand rows if the cell is rerun)
summary = pd.concat([summary[summary["Law"] != RAND_LAW], pd.DataFrame(rows)], ignore_index=True)
summary.to_csv(SUMMARY_CSV, index=False)
print("Saved:", os.path.relpath(SUMMARY_CSV), summary.shape)
print(summary[summary["Law"] == RAND_LAW][["Law", "Model", "CPC_mean", "CPL_mean", "CPCd_mean"]]
      .round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG4_NAME = "cz_lenormand_fig4_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
LAW_STYLE = {                    # facecolor, marker, legend label (as in the paper)
    "NGravExp": ("#ee6a5b", "o", "NGrav (exp)"),
    "NGravPow": ("#4b8bea", "s", "NGrav (pow)"),
    "Rad":      ("#a94fc0", "^", "Rad"),
}
PANELS = [("(a)", "CPC", "CPC"), ("(b)", "CPL", "CPL"), ("(c)", "CPCd", r"CPC$_d$")]
RAND_STYLE = {                   # metric -> (color, linestyle, label)
    "CPC": ("black", "-", "CPC"),
    "CPL": ("#666666", "--", "CPL"),
    "CPCd": ("#999999", ":", r"CPC$_d$"),
}
HEADROOM = 0.30                  # empty space above the data for the legend
# ------------------------------------------------------------------

x = list(range(len(CONSTRAINT_LEVELS)))

def pad_ylim(ax, values):
    lo, hi = min(values), max(values)
    span = (hi - lo) or 0.1
    ax.set_ylim(lo - 0.1 * span, hi + HEADROOM * span)

def style_axis(ax, title, ylabel):
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(CONSTRAINT_LEVELS, fontweight="bold")
    ax.set_xlabel("Constrained Models")
    ax.set_ylabel(ylabel)

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for ax, (title, metric, ylabel) in zip(axes.flat[:3], PANELS):
    all_vals = []
    for law, (color, marker, label) in LAW_STYLE.items():
        sub = summary[summary["Law"] == law].set_index("Model").loc[CONSTRAINT_LEVELS]
        y = sub[f"{metric}_mean"].to_numpy()
        all_vals += list(y)
        ax.plot(x, y, color="black", linewidth=1, zorder=2)
        ax.plot(x, y, linestyle="none", marker=marker, markersize=8, markerfacecolor=color,
                markeredgecolor="black", markeredgewidth=1, label=label, zorder=3)
    style_axis(ax, title, ylabel)
    pad_ylim(ax, all_vals)
    if metric == "CPC":
        ax.legend(loc="upper left", frameon=True, fontsize=9)

# panel (d): uniform distribution (Rand law)
ax = axes[1, 1]
rand = summary[summary["Law"] == "Rand"].set_index("Model").loc[CONSTRAINT_LEVELS]
vals = []
for metric, (color, ls, label) in RAND_STYLE.items():
    y = rand[f"{metric}_mean"].to_numpy()
    vals += list(y)
    ax.plot(x, y, color=color, linestyle=ls, marker="o", markersize=6, label=label)
style_axis(ax, "(d)", "Random")
pad_ylim(ax, vals)
ax.legend(loc="upper left", frameon=True, fontsize=9)

fig.tight_layout()
for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG4_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG4_NAME))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG5B_NAME = "cz_lenormand_fig5_4panel_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
PANEL_TITLES = {"UM": "Unconstrained Model", "PCM": "Production Constrained Model",
                "ACM": "Attraction Constrained Model", "DCM": "Doubly Constrained Model"}
LAW_Y_OFFSET = {"NGravExp": 0.2, "NGravPow": 0.0, "Rad": -0.2}   # separate the three laws vertically
XLIM = (0, 2)
# LAW_STYLE is defined in the Fig. 4 cell (color, marker, label)
# ------------------------------------------------------------------

links = pd.read_csv(LINKS_CSV)

fig, axes = plt.subplots(2, 2, figsize=(9, 5.5), sharex=True)
for ax, level in zip(axes.flat, CONSTRAINT_LEVELS):
    ax.axvline(1.0, color="black", linewidth=1, zorder=1)
    ax.axhline(0, color="#aaaaaa", linewidth=0.8, linestyle=":", zorder=0)
    for law, (color, marker, label) in LAW_STYLE.items():
        r = links[(links["Law"] == law) & (links["Model"] == level)].iloc[0]
        y = LAW_Y_OFFSET[law]
        # clip at 0: rounding can make mean - min a tiny negative number
        err_low = max(r["ratio_mean"] - r["ratio_min"], 0.0)
        err_high = max(r["ratio_max"] - r["ratio_mean"], 0.0)
        ax.errorbar(r["ratio_mean"], y, xerr=[[err_low], [err_high]],
                    color="black", linewidth=1, capsize=3, zorder=2)
        ax.plot(r["ratio_mean"], y, linestyle="none", marker=marker, markersize=9,
                markerfacecolor=color, markeredgecolor="black", markeredgewidth=1,
                label=label, zorder=3)
    ax.set_title(PANEL_TITLES[level], fontweight="bold")
    ax.set_yticks([0])
    ax.set_yticklabels([COUNTRY], fontweight="bold")
    ax.set_ylim(-0.5, 0.5)
    ax.set_xlim(*XLIM)
for ax in axes[1]:
    ax.set_xlabel("(#Simulated links) / (#Observed links)")

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 1))

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG5B_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG5B_NAME))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG6_LEVEL = "PCM"                      # the paper shows the production constrained model
BIN_KM = 2.0                            # bin width of the paper's CPCd definition
PDF_CSV = OUT_DIR / f"distance_pdf_{FIG6_LEVEL}.csv"
FIG6_NAME = f"cz_lenormand_fig6_{FIG6_LEVEL}_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
# LAW_STYLE is defined in the Fig. 4 cell (color, marker, label)
# ------------------------------------------------------------------

offdiag = ~np.eye(distance.shape[0], dtype=bool)
d_flat = distance[offdiag]
edges = np.arange(0, d_flat.max() + BIN_KM, BIN_KM)
centers = (edges[:-1] + edges[1:]) / 2

def distance_pdf(flows):
    """P(d): share of commuters per km in each distance bin (flows: n x n matrix)."""
    w = flows[offdiag]
    hist, _ = np.histogram(d_flat, bins=edges, weights=w)
    return hist / w.sum() / BIN_KM

pdf = pd.DataFrame({"d_km": centers, "Observed": distance_pdf(obs)})
for law in LAW_STYLE:
    sim = np.load(RUNS_DIR / f"sim_{law}_{FIG6_LEVEL}.npy")            # (repli, n, n)
    pdf[law] = np.mean([distance_pdf(s) for s in sim], axis=0)
pdf.to_csv(PDF_CSV, index=False)
print("Saved:", os.path.relpath(PDF_CSV), pdf.shape)
print("Non-empty bins (observed):", int((pdf["Observed"] > 0).sum()), "of", len(pdf))

fig, ax = plt.subplots(figsize=(6.5, 5))
obs_nz = pdf[pdf["Observed"] > 0]
ax.plot(obs_nz["d_km"], obs_nz["Observed"], color="black", marker="*", markersize=10,
        linewidth=1, label="Data", zorder=4)
for law, (color, marker, label) in LAW_STYLE.items():
    nz = pdf[pdf[law] > 0]
    ax.plot(nz["d_km"], nz[law], color=color, marker=marker, markersize=5,
            markeredgecolor="black", markeredgewidth=0.5, linewidth=0.8, label=label, zorder=3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("d (km)")
ax.set_ylabel("P(d)")
ax.set_title(f"Commuting distance distribution ({FIG6_LEVEL})", fontweight="bold")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=4)
fig.tight_layout()

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG6_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG6_NAME))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG6_LEVEL = "PCM"                      # the paper shows the production constrained model
BIN_KM = 20.0                           # coarser bins because only 182 distinct distances exist
PDF_CSV = OUT_DIR / f"distance_pdf_{FIG6_LEVEL}_bin{int(BIN_KM)}km.csv"
FIG6_NAME = f"cz_lenormand_fig6_{FIG6_LEVEL}_bin{int(BIN_KM)}km_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
# LAW_STYLE is defined in the Fig. 4 cell (color, marker, label)
# ------------------------------------------------------------------

offdiag = ~np.eye(distance.shape[0], dtype=bool)
d_flat = distance[offdiag]
edges = np.arange(0, d_flat.max() + BIN_KM, BIN_KM)
centers = (edges[:-1] + edges[1:]) / 2

def distance_pdf(flows):
    """P(d): share of commuters per km in each distance bin (flows: n x n matrix)."""
    w = flows[offdiag]
    hist, _ = np.histogram(d_flat, bins=edges, weights=w)
    return hist / w.sum() / BIN_KM

pdf = pd.DataFrame({"d_km": centers, "Observed": distance_pdf(obs)})
for law in LAW_STYLE:
    sim = np.load(RUNS_DIR / f"sim_{law}_{FIG6_LEVEL}.npy")            # (repli, n, n)
    pdf[law] = np.mean([distance_pdf(s) for s in sim], axis=0)
pdf.to_csv(PDF_CSV, index=False)
print("Saved:", os.path.relpath(PDF_CSV), pdf.shape)
print("Non-empty bins (observed):", int((pdf["Observed"] > 0).sum()), "of", len(pdf))

fig, ax = plt.subplots(figsize=(6.5, 5))
obs_nz = pdf[pdf["Observed"] > 0]
ax.plot(obs_nz["d_km"], obs_nz["Observed"], color="black", marker="*", markersize=10,
        linewidth=1, label="Data", zorder=4)
for law, (color, marker, label) in LAW_STYLE.items():
    nz = pdf[pdf[law] > 0]
    ax.plot(nz["d_km"], nz[law], color=color, marker=marker, markersize=5,
            markeredgecolor="black", markeredgewidth=0.5, linewidth=0.8, label=label, zorder=3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("d (km)")
ax.set_ylabel("P(d)")
ax.set_title(f"Commuting distance distribution ({FIG6_LEVEL}, {int(BIN_KM)} km bins)", fontweight="bold")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=4)
fig.tight_layout()

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG6_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG6_NAME))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG6_LEVEL = "PCM"                      # the paper shows the production constrained model
BIN_RATIO = 1.3                         # each bin is this many times wider than the previous one
PDF_CSV = OUT_DIR / f"distance_pdf_{FIG6_LEVEL}_logbin.csv"
FIG6_NAME = f"cz_lenormand_fig6_{FIG6_LEVEL}_logbin_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
# LAW_STYLE is defined in the Fig. 4 cell (color, marker, label)
# ------------------------------------------------------------------

offdiag = ~np.eye(distance.shape[0], dtype=bool)
d_flat = distance[offdiag]
# geometric bin edges covering the whole distance range
n_bins = int(np.ceil(np.log(d_flat.max() / d_flat.min()) / np.log(BIN_RATIO))) + 1
edges = d_flat.min() * 0.999 * BIN_RATIO ** np.arange(n_bins + 1)
widths = np.diff(edges)
centers = np.sqrt(edges[:-1] * edges[1:])

def distance_pdf(flows):
    """P(d): share of commuters per km in each (log-spaced) distance bin."""
    w = flows[offdiag]
    hist, _ = np.histogram(d_flat, bins=edges, weights=w)
    return hist / w.sum() / widths

pdf = pd.DataFrame({"d_km": centers, "bin_width_km": widths, "Observed": distance_pdf(obs)})
for law in LAW_STYLE:
    sim = np.load(RUNS_DIR / f"sim_{law}_{FIG6_LEVEL}.npy")            # (repli, n, n)
    pdf[law] = np.mean([distance_pdf(s) for s in sim], axis=0)
pdf.to_csv(PDF_CSV, index=False)
print("Saved:", os.path.relpath(PDF_CSV), pdf.shape)
print("Non-empty bins (observed):", int((pdf["Observed"] > 0).sum()), "of", len(pdf))

fig, ax = plt.subplots(figsize=(6.5, 5))
obs_nz = pdf[pdf["Observed"] > 0]
ax.plot(obs_nz["d_km"], obs_nz["Observed"], color="black", marker="*", markersize=10,
        linewidth=1, label="Data", zorder=4)
for law, (color, marker, label) in LAW_STYLE.items():
    nz = pdf[pdf[law] > 0]
    ax.plot(nz["d_km"], nz[law], color=color, marker=marker, markersize=5,
            markeredgecolor="black", markeredgewidth=0.5, linewidth=0.8, label=label, zorder=3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("d (km)")
ax.set_ylabel("P(d)")
ax.set_title(f"Commuting distance distribution ({FIG6_LEVEL}, log bins x{BIN_RATIO})", fontweight="bold")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=4)
fig.tight_layout()

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG6_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG6_NAME))
plt.show()

In [ ]:
import io
import warnings
from contextlib import redirect_stdout

# ----------------------------- CONFIG -----------------------------
IO_SUFFIX = "flows"                                   # inputs: O_i and D_j instead of population
RUNS_DIR_IO = OUT_DIR / f"runs_{IO_SUFFIX}"
CALIB_IO_CSV = OUT_DIR / f"calibrated_exponents_{IO_SUFFIX}.csv"
SUMMARY_IO_CSV = OUT_DIR / f"metrics_pytdlm_summary_{IO_SUFFIX}.csv"
IO_LAWS = ["NGravExp", "NGravPow", "Rad"]
# N_REPS, SEED, SEED_EVAL, N_REPS_CALIB, CALIB_MEASURE, CONSTRAINT_LEVELS, METRIC_COLS,
# first_value come from earlier cells
# ------------------------------------------------------------------

RUNS_DIR_IO.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# 1) calibration of the exponents with O_i and D_j as masses
if CALIB_IO_CSV.exists():
    calib_io = pd.read_csv(CALIB_IO_CSV)
    print("Loaded from checkpoint:", os.path.relpath(CALIB_IO_CSV))
else:
    with redirect_stdout(io.StringIO()):
        calib_io = tdlm.run_optimization(
            mass_origin=out_trips, mass_destination=in_trips, distance=distance, obs=obs,
            law=["NGravExp", "NGravPow"], model=CONSTRAINT_LEVELS,
            out_trips=out_trips, in_trips=in_trips,
            repli=N_REPS_CALIB, measure=CALIB_MEASURE, random_seed=SEED)
    calib_io.to_csv(CALIB_IO_CSV, index=False)
    print("Saved:", os.path.relpath(CALIB_IO_CSV))
print(calib_io.round(4).to_string(index=False))

# 2) final runs and goodness-of-fit measures (one checkpoint per model)
frames = []
for law in IO_LAWS:
    for level in CONSTRAINT_LEVELS:
        tag = f"{law}_{level}"
        sim_path, met_path = RUNS_DIR_IO / f"sim_{tag}.npy", RUNS_DIR_IO / f"metrics_{tag}.csv"
        if not (sim_path.exists() and met_path.exists()):
            if law == "Rad":
                exponent = 1.0                       # ignored by the radiation law
            else:
                row = calib_io[(calib_io["Law"] == law) & (calib_io["Model"] == level)].iloc[0]
                exponent = np.array([row["Exponent"]])
            with redirect_stdout(io.StringIO()):
                sim = tdlm.run_law_model(
                    law=law, mass_origin=out_trips, mass_destination=in_trips, distance=distance,
                    exponent=exponent, model=level, out_trips=out_trips, in_trips=in_trips,
                    repli=N_REPS, random_seed=SEED_EVAL, verbose=False)
                res = tdlm.gof(sim=sim, obs=obs, distance=distance)
            first_value(res)[["Replication"] + METRIC_COLS].to_csv(met_path, index=False)
            np.save(sim_path, np.asarray(first_value(sim)))
            print("Done:", tag)
        else:
            print("Skip (checkpoint):", tag)
        m = pd.read_csv(met_path)
        m.insert(0, "Model", level)
        m.insert(0, "Law", law)
        frames.append(m)

by_rep_io = pd.concat(frames, ignore_index=True)
summary_io = by_rep_io.groupby(["Law", "Model"], sort=False)[METRIC_COLS].agg(["mean", "std"])
summary_io.columns = [f"{a}_{b}" for a, b in summary_io.columns]
summary_io = summary_io.reset_index()
summary_io.to_csv(SUMMARY_IO_CSV, index=False)
print("Saved:", os.path.relpath(SUMMARY_IO_CSV), summary_io.shape)
print(summary_io[["Law", "Model", "CPC_mean", "CPL_mean", "CPCd_mean", "KL_div_mean", "RMSE_mean"]]
      .round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
FIG7_NAME = "cz_lenormand_fig7_pytdlm.png"
FIG_DIRS = [Path("../figures"), Path("../plots")]
PANELS7 = [("(a)", "CPC", "CPC"), ("(b)", "CPL", "CPL"), ("(c)", "CPCd", r"CPC$_d$")]
HEADROOM = 0.15
# LAW_STYLE is defined in the Fig. 4 cell (color, marker, label)
# ------------------------------------------------------------------

x = list(range(len(CONSTRAINT_LEVELS)))
fig, axes = plt.subplots(1, 3, figsize=(12, 4.3))
for ax, (title, metric, ylabel) in zip(axes, PANELS7):
    vals = []
    for law, (color, marker, label) in LAW_STYLE.items():
        sub = summary_io[summary_io["Law"] == law].set_index("Model").loc[CONSTRAINT_LEVELS]
        y = sub[f"{metric}_mean"].to_numpy()
        vals += list(y)
        ax.plot(x, y, color="black", linewidth=1, zorder=2)
        ax.plot(x, y, linestyle="none", marker=marker, markersize=8, markerfacecolor=color,
                markeredgecolor="black", markeredgewidth=1, label=label, zorder=3)
    span = (max(vals) - min(vals)) or 0.1
    ax.set_ylim(min(vals) - 0.1 * span, max(vals) + HEADROOM * span)
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(CONSTRAINT_LEVELS, fontweight="bold")
    ax.set_xlabel("Constrained Models")
    ax.set_ylabel(ylabel)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.07, 1, 1))

for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / FIG7_NAME, dpi=200, facecolor="white")
    print("Saved:", os.path.relpath(d / FIG7_NAME))
plt.show()